![Banner](https://i.imgur.com/a3uAqnb.png)

# Implement Oriented Bounding Box (OBB) Detection Using YOLO from scratch - Homework Assignment

![Architecture of the YOLOv11-OBB](https://www.researchgate.net/publication/393366901/figure/fig2/AS:11431281539704197@1752249585792/Architecture-of-the-YOLOv11-OBB-model.tif)

In this assignment, you will create a dataset class for a subset of the Dota OBB dataset and use oriented YOLO from ultralytics for detecting objects with oriented bounding boxes.

## 📌 Project Overview
- **Task**: Multi-class oriented object detection using DOTA dataset subset
- **Architecture**: YOLO variants with Oriented Bounding Box (OBB) detection
- **Dataset**: DOTA Demo dataset (58 images) from Roboflow
- **Goal**: Compare performance of different YOLO models for OBB detection

## 📚 Learning Objectives
By completing this assignment, you will:
- Understand oriented bounding box detection vs regular bounding boxes
- Learn about YOLO architecture variants (YOLOv8n, YOLOv8m, YOLO11n)
- Implement custom dataset classes for OBB data format
- Compare multiple model architectures on the same task
- Evaluate OBB detection models using specialized metrics
- Visualize oriented bounding box predictions

## 🎯 Evaluation Metrics
You will be evaluated using:
- **mAP@0.5**: Mean Average Precision at IoU threshold 0.5 for OBB
- **mAP@0.5:0.95**: Mean Average Precision across IoU thresholds 0.5-0.95
- **Precision**: Overall precision across all classes
- **Recall**: Overall recall across all classes
- **F1-Score**: Harmonic mean of precision and recall

## Dataset:
You will use a subset of DOTA Dataset, consisting of 58 images. Dataset can be downloaded from:
https://universe.roboflow.com/rotated-object-detection/dota-demo/dataset/1

## Tasks:
1. Create a dataset class for the above dataset
2. Select 3 different YOLO models from ultralytics (different YOLO versions)
3. Train all models on the training dataset
4. Evaluate all models on the test set
5. Plot predictions for all models on the same images

## 1️⃣ Dataset Setup and Library Imports

**Task**: Import necessary libraries and download the DOTA OBB dataset from Roboflow.

**Requirements**:
- Import all required libraries for computer vision, deep learning, and visualization
- Set up matplotlib configuration for consistent plotting
- Configure Roboflow API access with your personal API key
- Download the DOTA Demo dataset in YOLO OBB format
- Parse the dataset configuration file (data.yaml)


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import yaml
import json
from roboflow import Roboflow
import torch
from ultralytics import YOLO
import pandas as pd
from PIL import Image
import shutil

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

rf = Roboflow(api_key="KEK")
project = rf.workspace("rotated-object-detection").project("dota-demo")
dataset = project.version(1).download("yolov8-obb")

dataset_path = Path("./DOTA-Demo-1")
data_yaml_path = dataset_path / "data.yaml"

with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

## 2️⃣ Custom Dataset Class Implementation

**Task**: Create a comprehensive PyTorch Dataset class for DOTA OBB format.

**Requirements**:
- Inherit from torch.utils.data.Dataset
- Parse OBB annotations in YOLO format (normalized coordinates)
- Convert between normalized and pixel coordinates
- Implement visualization methods for oriented bounding boxes
- Handle different image formats (jpg, png) if needed
- Provide proper data validation and error handling

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from typing import List, Tuple, Dict
import random

class DOTAOBBDataset(Dataset):
    def __init__(self, images_dir: str, labels_dir: str, class_names: List[str], transform=None):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.class_names = class_names
        self.num_classes = len(class_names)
        self.transform = transform
        
        self.image_files = list(self.images_dir.glob('*.jpg')) + list(self.images_dir.glob('*.png'))
        self.image_files.sort()
        
        self.valid_samples = []
        for img_file in self.image_files:
            label_file = self.labels_dir / (img_file.stem + '.txt')
            if label_file.exists():
                self.valid_samples.append((img_file, label_file))
    
    def __len__(self):
        return len(self.valid_samples)
    
    def __getitem__(self, idx):
        img_path, label_path = self.valid_samples[idx]
        
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]
        
        annotations = self.load_obb_annotations(label_path, w, h)
        
        if self.transform:
            image = self.transform(image)
        
        return {
            'image': image,
            'annotations': annotations,
            'image_path': str(img_path),
            'image_size': (w, h)
        }
    
    def load_obb_annotations(self, label_path: Path, img_w: int, img_h: int) -> List[Dict]:
        annotations = []
        
        with open(label_path, 'r') as f:
            lines = f.readlines()
        
        for line in lines:
            line = line.strip()
            if not line:
                continue
            
            parts = line.split()
            if len(parts) < 9:
                continue
            
            class_id = int(parts[0])
            coords = [float(x) for x in parts[1:9]]
            
            pixel_coords = []
            for i in range(0, 8, 2):
                x = coords[i] * img_w
                y = coords[i+1] * img_h
                pixel_coords.extend([x, y])
            
            annotation = {
                'class_id': class_id,
                'class_name': self.class_names[class_id],
                'obb_coords': coords,
                'pixel_coords': pixel_coords,
            }
            annotations.append(annotation)
        
        return annotations
    
    def visualize_sample(self, idx: int, figsize=(12, 8)):
        sample = self[idx]
        image = sample['image']
        annotations = sample['annotations']
        
        if isinstance(image, torch.Tensor):
            image = image.permute(1, 2, 0).numpy()
        
        if image.dtype != np.uint8:
            image = (image * 255).astype(np.uint8)
        
        plt.figure(figsize=figsize)
        plt.imshow(image)
        
        colors = ['red', 'blue', 'green', 'yellow', 'orange', 'purple', 'pink', 'brown']
        
        for ann in annotations:
            coords = ann['pixel_coords']
            class_name = ann['class_name']
            
            points = [(coords[i], coords[i+1]) for i in range(0, 8, 2)]
            points.append(points[0])
            
            color = colors[ann['class_id'] % len(colors)]
            
            xs, ys = zip(*points)
            plt.plot(xs, ys, color=color, linewidth=2, alpha=0.8)
            plt.fill(xs, ys, color=color, alpha=0.2)
            
            center_x = sum(coords[::2]) / 4
            center_y = sum(coords[1::2]) / 4
            plt.text(center_x, center_y, class_name, 
                    fontsize=10, color='white', weight='bold',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor=color, alpha=0.7))
        
        plt.title(f"Sample {idx}: {Path(sample['image_path']).name}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()

train_dataset = DOTAOBBDataset(
    images_dir=dataset_path / "train" / "images",
    labels_dir=dataset_path / "train" / "labels", 
    class_names=list(data_config['names'].values())
)

test_dataset = DOTAOBBDataset(
    images_dir=dataset_path / "test" / "images",
    labels_dir=dataset_path / "test" / "labels",
    class_names=list(data_config['names'].values())
)

valid_dataset = DOTAOBBDataset(
    images_dir=dataset_path / "valid" / "images",
    labels_dir=dataset_path / "valid" / "labels",
    class_names=list(data_config['names'].values())
)

for i in range(min(3, len(train_dataset))):
    train_dataset.visualize_sample(i)


## 3️⃣ Model Training and Comparison

**Task**: Train and compare multiple YOLO models for OBB detection.

**Requirements**:
- Select 3 different YOLO model variants from Ultralytics
- Configure training hyperparameters consistently across models
- Track training progress and performance metrics
- Save trained models for evaluation and comparison
- Monitor computational efficiency and training time

In [ ]:
import time
from ultralytics import YOLO
import gc
import torch

results_dir = Path("./results")
results_dir.mkdir(exist_ok=True)

EPOCHS = 50
BATCH_SIZE = 8
IMG_SIZE = 640

models_config = {
    'yolo11n-obb': {
        'model_name': 'yolo11n-obb.pt',
        'description': 'YOLO11 Nano OBB', 
        'save_dir': results_dir / 'yolo11n_obb'
    },
    'yolo8n-obb': {
        'model_name': 'yolov8n-obb.pt',
        'description': 'YOLOv8 Nano OBB',
        'save_dir': results_dir / 'yolo8n_obb'
    },
    'yolo8m-obb': {
        'model_name': 'yolov8m-obb.pt', 
        'description': 'YOLOv8 Medium OBB',
        'save_dir': results_dir / 'yolo8m_obb'
    }
}

training_results = {}

for model_key, config in models_config.items():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
    
    model = YOLO(config['model_name'])
    config['save_dir'].mkdir(exist_ok=True, parents=True)
    
    start_time = time.time()
    
    results = model.train(
        data=str(data_yaml_path),
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        project=str(results_dir),
        name=model_key,
        exist_ok=True,
        patience=0,
        save=True,
        plots=True,
        verbose=False
    )
    
    training_time = time.time() - start_time
    
    training_results[model_key] = {
        'model': model,
        'results': results,
        'training_time': training_time,
        'config': config,
        'model_path': results_dir / model_key / 'weights' / 'best.pt'
    }

## 4️⃣ Model Evaluation and Visualization

**Task**: Evaluate all trained models and create comprehensive visualizations.

**Requirements**:
- Load best trained weights for each model
- Evaluate performance on test dataset using consistent metrics  
- Create performance comparison charts and plots
- Visualize predictions on test images with ground truth overlay
- Generate per-class performance analysis
- Compare computational efficiency across models

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Polygon
import numpy as np
from collections import defaultdict
import time

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

trained_models = {}
model_names = ['yolo11n-obb', 'yolo8n-obb', 'yolo8m-obb']

for model_key in model_names:
    model_path = results_dir / model_key / 'weights' / 'best.pt'
    if model_path.exists():
        trained_models[model_key] = YOLO(str(model_path))

evaluation_results = {}

for model_key, model in trained_models.items():
    start_time = time.time()
    results = model.val(
        data=str(data_yaml_path),
        split='test',
        device='cuda' if torch.cuda.is_available() else 'cpu',
        plots=False,
        verbose=False
    )
    eval_time = time.time() - start_time
    
    metrics = {
        'mAP50': results.box.map50,
        'mAP50-95': results.box.map,
        'Precision': results.box.mp,
        'Recall': results.box.mr,
        'F1': 2 * (results.box.mp * results.box.mr) / (results.box.mp + results.box.mr) if (results.box.mp + results.box.mr) > 0 else 0,
        'Inference_Time': eval_time,
        'mAP_per_class': results.box.maps.tolist() if results.box.maps is not None else []
    }
    
    evaluation_results[model_key] = metrics

# Model Comparison Bar Plot
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

metrics_to_plot = ['mAP50', 'mAP50-95', 'Precision', 'Recall', 'F1', 'Inference_Time']
metric_titles = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall', 'F1-Score', 'Inference Time (s)']

for idx, (metric, title) in enumerate(zip(metrics_to_plot, metric_titles)):
    ax = axes[idx // 3, idx % 3]
    
    models = list(evaluation_results.keys())
    values = [evaluation_results[model][metric] for model in models]
    display_names = [name.replace('-obb', '') for name in models]
    
    bars = ax.bar(display_names, values, alpha=0.8, edgecolor='black', linewidth=1)
    
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    for bar, color in zip(bars, colors):
        bar.set_color(color)
    
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(title)
    ax.tick_params(axis='x', rotation=45)
    
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                f'{value:.4f}' if metric != 'Inference_Time' else f'{value:.2f}s',
                ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Per-Class Performance Heatmap
if any(evaluation_results[model]['mAP_per_class'] for model in evaluation_results):
    class_names = list(data_config['names'].values())
    
    per_class_data = []
    for model_key in trained_models.keys():
        if evaluation_results[model_key]['mAP_per_class']:
            per_class_data.append(evaluation_results[model_key]['mAP_per_class'])
    
    if per_class_data:
        per_class_matrix = np.array(per_class_data)
        
        plt.figure(figsize=(12, 8))
        sns.heatmap(per_class_matrix, 
                   xticklabels=class_names,
                   yticklabels=[name.replace('-obb', '') for name in trained_models.keys()],
                   annot=True, fmt='.3f', cmap='YlOrRd',
                   cbar_kws={'label': 'mAP@0.5'})
        plt.title('Per-Class mAP@0.5 Performance Heatmap', fontsize=14, fontweight='bold')
        plt.xlabel('Classes', fontweight='bold')
        plt.ylabel('Models', fontweight='bold')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

# Model Performance Radar Chart
def create_radar_chart(evaluation_results):
    metrics = ['mAP50', 'mAP50-95', 'Precision', 'Recall', 'F1']
    
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))
    
    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False)
    angles = np.concatenate((angles, [angles[0]]))
    
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    
    for idx, (model_key, color) in enumerate(zip(trained_models.keys(), colors)):
        values = [evaluation_results[model_key][metric] for metric in metrics]
        values += [values[0]]
        
        ax.plot(angles, values, 'o-', linewidth=2, label=model_key.replace('-obb', ''), color=color)
        ax.fill(angles, values, alpha=0.25, color=color)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics, fontsize=12)
    ax.set_ylim(0, 1)
    ax.set_title('Model Performance Radar Chart', fontsize=16, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
    ax.grid(True)
    
    plt.tight_layout()
    plt.show()

create_radar_chart(evaluation_results)

# Prediction Visualization on Test Images
def plot_predictions_comparison(image_idx, conf_threshold=0.3):
    test_sample = test_dataset[image_idx]
    image_path = test_sample['image_path']
    
    original_image = cv2.imread(image_path)
    original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 16))
    
    ax = axes[0, 0]
    ax.imshow(original_image)
    ax.set_title('Ground Truth', fontsize=14, fontweight='bold')
    
    colors = ['red', 'blue', 'green', 'yellow', 'orange']
    for ann in test_sample['annotations']:
        coords = ann['pixel_coords']
        points = [(coords[i], coords[i+1]) for i in range(0, 8, 2)]
        polygon = Polygon(points, fill=False, edgecolor=colors[ann['class_id'] % len(colors)], 
                         linewidth=2, alpha=0.8)
        ax.add_patch(polygon)
        
        center_x = sum(coords[::2]) / 4
        center_y = sum(coords[1::2]) / 4
        ax.text(center_x, center_y, ann['class_name'], 
               fontsize=8, color='white', weight='bold',
               bbox=dict(boxstyle="round,pad=0.3", facecolor=colors[ann['class_id'] % len(colors)], alpha=0.7))
    
    ax.axis('off')
    
    model_positions = [(0, 1), (1, 0), (1, 1)]
    
    for idx, (model_key, pos) in enumerate(zip(trained_models.keys(), model_positions)):
        ax = axes[pos[0], pos[1]]
        ax.imshow(original_image)
        ax.set_title(f'{model_key.replace("-obb", "")} Predictions', fontsize=14, fontweight='bold')
        
        model = trained_models[model_key]
        results = model.predict(image_path, conf=conf_threshold, device='cuda' if torch.cuda.is_available() else 'cpu')
        
        if len(results) > 0 and results[0].obb is not None:
            boxes = results[0].obb
            
            for i in range(len(boxes.xyxyxyxy)):
                obb_coords = boxes.xyxyxyxy[i].cpu().numpy()
                confidence = boxes.conf[i].cpu().numpy()
                class_id = int(boxes.cls[i].cpu().numpy())
                class_name = data_config['names'][class_id]
                
                polygon = Polygon(obb_coords, fill=False, edgecolor=colors[class_id % len(colors)], 
                                linewidth=2, alpha=0.8)
                ax.add_patch(polygon)
                
                center_x = np.mean(obb_coords[:, 0])
                center_y = np.mean(obb_coords[:, 1])
                ax.text(center_x, center_y, f'{class_name}\n{confidence:.2f}', 
                       fontsize=8, color='white', weight='bold',
                       bbox=dict(boxstyle="round,pad=0.3", facecolor=colors[class_id % len(colors)], alpha=0.7),
                       ha='center')
        
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

num_test_images_to_show = min(3, len(test_dataset))
for i in range(num_test_images_to_show):
    plot_predictions_comparison(i, conf_threshold=0.3)

# Performance Summary Table
df = pd.DataFrame(evaluation_results).T
df.index = [name.replace('-obb', '') for name in df.index]
df = df[['mAP50', 'mAP50-95', 'Precision', 'Recall', 'F1', 'Inference_Time']]
df.columns = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall', 'F1-Score', 'Inference Time (s)']
print(df.round(4))

## 📝 Evaluation Criteria

Your homework will be evaluated based on:

### 1. Implementation Correctness (40%)
- **Dataset Class**: Proper OBB annotation parsing and coordinate conversion
- **YOLO Integration**: Correct usage of Ultralytics YOLO models for OBB detection
- **Training Pipeline**: Working training loop with proper hyperparameter configuration
- **Evaluation Metrics**: Accurate implementation of mAP calculation for oriented boxes
- **Visualization**: Correct display of oriented bounding boxes and predictions

### 2. Model Training and Performance (30%)
- **Multi-Model Training**: Successfully training all 3 YOLO variants
- **Convergence**: Reasonable training progress and metric improvement
- **Performance Analysis**: Meaningful comparison between different model architectures
- **Results Quality**: Achieving acceptable detection performance on test set
- **Efficiency Analysis**: Proper comparison of training time and inference speed

### 3. Code Quality and Analysis (20%)
- **Code Structure**: Clean, readable code with proper organization
- **Documentation**: Adequate comments explaining OBB concepts and implementation
- **Comparative Analysis**: Thoughtful comparison of model performance and trade-offs

### 4. Visualization and Presentation (10%)
- **Chart Quality**: Professional and informative performance comparison plots
- **Prediction Visualization**: Clear display of oriented bounding box predictions
- **Results Interpretation**: Meaningful analysis of model strengths and weaknesses
- **Technical Communication**: Clear explanation of OBB detection concepts

## 🎯 Expected Performance Benchmarks
- **Minimum mAP@0.5**: > 0.30 (at least one model should achieve this)
- **Training Completion**: All models should train without errors
- **Visualization Quality**: Clear, labeled plots with proper oriented box rendering
- **Comparative Analysis**: Identify best performing model with justification


## Hard bonus question:
 Implement one of them from scratch, this will take you a while to get decent outputs(You will 5 extra points if your from sctrach model gets mAP@0.5: > 0.30. so if you get 10 you will get 15, if you get 8 you will get 13 and so on.)